ISMB 2024 Tutorial: Multi-omic data integration for microbiome research using scikit-bio

# Merging abundance with taxonomy
Merging abundace with taxonomy in bioms

Goals
   - [x] Merging abundance with taxonomy in bioms

## Preparation

Install the latest version of scikit-bio if it hasn't been (needed for every Google Colab instance).

In [18]:
from importlib.util import find_spec

In [19]:
if find_spec('skbio') is None:
    !pip install -q scikit-bio

In [20]:
import skbio
skbio.__version__

'0.7.1'

Import common libraries.

In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

If you use Google Colab, and would like to directly mount the shared Google Drive folder containing data files, please uncomment and execute the following code.

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
# # Specify your remote directory
HOME = '/content/drive/MyDrive/sci-kit-bio_local/Data/emp500'

If you use Google Colab or local Jupyter, and would like to download the data file package to the current directory, please uncomment and execute the following code.

If you use local Jupyter, and have already downloaded and extracted the data file package, please specify its directory.

In [24]:
# # Specify your local directory
#HOME = '/home/drz/Desktop/Data/emp500'

Check if the directory exists by listing its content.

In [25]:
!ls $HOME

amplicon  assembly  masspec  README.md	sample.tsv  shotgun


## The EMP500 study

A `README.md` file within the data directory provides basic information about the study. Take a quick look at it:

## Sample metadata

### Metadata table

The sample metadata table, `sample.tsv`, is a tab-separated values ([TSV](https://en.wikipedia.org/wiki/Tab-separated_values)) file with samples as rows and properties as columns. Let's take a peek at the table:

In [26]:
!head $HOME/sample.tsv | cut -c1-100

sample_name	sample_name_original	sample_type	collection_timestamp	country	depth_sample	description	e
13114.angenent.65.s001	Angenent65.misc.104	bioreactor sludge	08/09/2011 0:00	USA - New York	not appl
13114.angenent.65.s002	Angenent65.misc.519	bioreactor sludge	9/27/12 0:00	USA - New York	not applica
13114.angenent.65.s003	Angenent65.misc.946	bioreactor sludge	11/28/13 0:00	USA - New York	not applic
13114.angenent.65.s004	Angenent65.misc.1022	bioreactor sludge	02/12/2014 0:00	USA - New York	not app
13114.angenent.65.s005	Angenent65.misc.1538	bioreactor sludge	7/13/15 0:00	USA - New York	not applic
13114.angenent.65.s006	Angenent65.misc.1622	bioreactor sludge	10/05/2015 0:00	USA - New York	not app
13114.angenent.65.s007	Angenent65.misc.1722	bioreactor sludge	1/13/15 0:00	USA - New York	not applic
13114.angenent.65.s008	Angenent65.misc.1818	bioreactor sludge	4/18/16 0:00	USA - New York	not applic
13114.angenent.65.s009	Angenent65.misc.1888	bioreactor sludge	6/27/16 0:00	USA - New York	n

In [27]:
from skbio.metadata import SampleMetadata

In [28]:
meta = SampleMetadata.load(f'{HOME}/sample.tsv', default_missing_scheme='INSDC:missing')
#meta

A `SampleMetadata` object can be converted into a Pandas dataframe with:

In [29]:
df_meta = meta.to_dataframe()
df_meta.head(100).tail(5)

,sample_name_original,sample_type,collection_timestamp,country,depth_sample,description,elevation,emp500_principal_investigator,emp500_study_id,emp500_pi_study_id,...,env_feature,env_material,env_package,geo_loc_name,host_subject_id,host_scientific_name,latitude,longitude,project_name,scientific_name
sample_name,,,,,,,,,,,,,,,,,,,,,
13114.jensen.43.s006,Jensen43.BZ15.16,sediment,09/06/2015,Belize,6,Jensen43.sediment.7,0.0,Jensen,43.0,Jensen43,...,lagoon,marine sediment,sediment,Atlantic Ocean:Northwest Atlantic Ocean:Belize,BZ15-16,NaN,16.79556,-88.1225,Jensen sediments from global ocean,marine sediment metagenome
13114.jensen.43.s007,Jensen43.MCO300.25.2,sediment,07/08/2012,USA - California,300,Jensen43.sediment.36,0.0,Jensen,43.0,Jensen43,...,mud,marine sediment,sediment,Pacific Ocean:Northeast Pacific Ocean:San Dieg...,MCO300-25.2,NaN,32.81000,-117.4681,Jensen sediments from global ocean,marine sediment metagenome
13114.jensen.43.s008,Jensen43.MCO700.17.1,sediment,07/05/2012,USA - California,700,Jensen43.sediment.31,0.0,Jensen,43.0,Jensen43,...,mud,marine sediment,sediment,Pacific Ocean:Northeast Pacific Ocean:San Dieg...,MCO700-17.1,NaN,32.81000,-117.4509,Jensen sediments from global ocean,marine sediment metagenome
13114.jensen.43.s009,Jensen43.MCO700.17.2,sediment,07/05/2012,USA - California,700,Jensen43.sediment.32,0.0,Jensen,43.0,Jensen43,...,mud,marine sediment,sediment,Pacific Ocean:Northeast Pacific Ocean:San Dieg...,MCO700-17.2,NaN,32.81000,-117.4509,Jensen sediments from global ocean,marine sediment metagenome
13114.jensen.43.s010,Jensen43.MCO700.17.3,sediment,07/05/2012,USA - California,700,Jensen43.sediment.33,0.0,Jensen,43.0,Jensen43,...,mud,marine sediment,sediment,Pacific Ocean:Northeast Pacific Ocean:San Dieg...,MCO700-17.3,NaN,32.81000,-117.4509,Jensen sediments from global ocean,marine sediment metagenome


In [30]:
# Get the shape of df_meta
df_meta_shape = df_meta.shape

# Print the shape
print(f"The shape of df_meta is: {df_meta_shape}")

The shape of df_meta is: (880, 33)


So there are metadata for 880 samples. In the next section we will filter to keep metadata only of those samples that have a minimal number of reads after quality control.

Here we can test metadata for selected samples.

In [31]:
# Filter the df_meta DataFrame for samples where 'empo_3' is 'Surface Saline'
surface_saline_samples = df_meta[df_meta['empo_3'] == 'Surface (saline)']

# Display the filtered DataFrame
print("Samples with EMPO 3 category 'Surface Saline':")
#display(surface_saline_samples)

# Print the row names (sample IDs) of the filtered DataFrame
#print("\nSample IDs with EMPO 3 category 'Surface Saline':")
print(surface_saline_samples.index.tolist())

# Filter the df_meta DataFrame for samples where 'empo_3' is 'Surface Saline'
APG_samples = df_meta[df_meta['empo_3'] == 'Animal proximal gut']

# Display the filtered DataFrame
print("Samples with EMPO 3 category 'Animal proximal gut':")
#display(surface_saline_samples)

# Print the row names (sample IDs) of the filtered DataFrame
#print("\nSample IDs with EMPO 3 category 'Surface Saline':")
print(APG_samples.index.tolist())

Samples with EMPO 3 category 'Surface Saline':
['13114.distel.72.s014', '13114.distel.72.s015', '13114.rohwer.86.s001', '13114.rohwer.86.s003']
Samples with EMPO 3 category 'Animal proximal gut':
['13114.angenent.65.s001', '13114.angenent.65.s002', '13114.angenent.65.s003', '13114.angenent.65.s004', '13114.angenent.65.s005', '13114.angenent.65.s006', '13114.angenent.65.s007', '13114.angenent.65.s008', '13114.angenent.65.s009', '13114.distel.72.s007', '13114.distel.72.s008', '13114.distel.72.s009', '13114.distel.72.s011', '13114.distel.72.s012', '13114.distel.72.s013', '13114.sandin.54.s001', '13114.sandin.54.s002', '13114.sandin.54.s003', '13114.sandin.54.s004', '13114.sandin.54.s005', '13114.sandin.54.s006', '13114.sandin.54.s007', '13114.sandin.54.s008', '13114.sandin.54.s009', '13114.sandin.54.s010', '13114.sandin.54.s011', '13114.sandin.54.s012', '13114.sandin.54.s013', '13114.sandin.54.s014', '13114.sandin.54.s015']


In [32]:
# Select the row for the specific sample and the 'empo' columns
empo_data_for_sample = df_meta.loc['13114.angenent.65.s001', ['empo_1', 'empo_2', 'empo_3']]

# Display the selected data
print("EMPO categories for sample 13114.zaneveld.9.s021:")
display(empo_data_for_sample)

EMPO categories for sample 13114.zaneveld.9.s021:


,13114.angenent.65.s001
empo_1,Host-associated
empo_2,Animal
empo_3,Animal proximal gut


## Shotgun sequencing kraken analysis

This tsv already in github is the result of kraken-bracken analysis (alberto), plus test if every taxon isyeast or no (Elaine script)

In [33]:
import pandas as pd

# GitHub raw file URL
github_tsv_url = 'https://raw.githubusercontent.com/nselem/2025LitiVisit/refs/heads/main/code/kraken_absolute_isyeast.tsv'

# Load the TSV file into a pandas DataFrame
try:
    yeast_abundance_df = pd.read_csv(github_tsv_url, sep='\t')

    # Display the head of the DataFrame to verify
    display(yeast_abundance_df.head(2))

except Exception as e:
    print(f"An error occurred: {e}")
    print("Please ensure the GitHub link is correct and the file is accessible.")

,taxid,13114.angenent.65.s001,13114.angenent.65.s002,13114.angenent.65.s003,13114.angenent.65.s004,13114.angenent.65.s005,13114.angenent.65.s006,13114.angenent.65.s007,13114.angenent.65.s008,13114.angenent.65.s009,...,phylum,class,order,family,genus,species,subphylum,taxa,result,isyeast
0,2036908,8922.0,0.0,0.0,0.0,0.0,0.0,101.0,0.0,0.0,...,Ascomycota,Leotiomycetes,Helotiales,Rutstroemiaceae,Clarireedia,jacksonii,Pezizomycotina,Clarireedia,No results found,N
1,2860823,4241.0,0.0,0.0,0.0,231.0,0.0,0.0,0.0,0.0,...,Ascomycota,Leotiomycetes,Helotiales,Rutstroemiaceae,Clarireedia,paspali,Pezizomycotina,Clarireedia,No results found,N


In [41]:
import plotly.graph_objects as go

# Select the data for the specific sample
#13114.distel.72.s014
#13114.rohwer.86.s001

#sample_id = '13114.angenent.65.s002'
sample_data = yeast_abundance_df[['taxid', 'phylum', 'class', 'order', 'family', 'genus', 'species', sample_id]].copy()
#print(yeast_abundance_df[['taxid', 'kingdom','phylum', 'class', 'order', 'family', 'genus', 'species', sample_id]][1:5])
#print("Sample data")
print(sample_data )

# Filter out rows where the abundance for the sample is 0 or NaN
sample_data = sample_data[sample_data[sample_id] > 0].dropna(subset=[sample_id])

# Drop rows where any of the taxonomy levels are missing or empty strings
sample_data = sample_data.replace('', np.nan).dropna(subset=['taxid', 'phylum', 'class', 'order', 'family', 'genus', 'species'])



# Create the Sankey plot data
labels = []
source = []
target = []
value = []

# Helper function to add nodes and links
def add_node_and_link(level1, level2, current_value):
    if level1 not in labels:
        labels.append(level1)
    if level2 not in labels:
        labels.append(level2)
    source.append(labels.index(level1))
    target.append(labels.index(level2))
    value.append(current_value)

# Iterate through the sample data and build the Sankey components
for index, row in sample_data.iterrows():
    abundance = row[sample_id]
    taxonomy = [row['phylum'], row['class'], row['order'], row['family'], row['genus'], row['species']]

    for i in range(len(taxonomy) - 1):
        level1 = taxonomy[i]
        level2 = taxonomy[i+1]
        add_node_and_link(level1, level2, abundance)

# Create the Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
        color="blue"
    ),
    link=dict(
        source=source,
        target=target,
        value=value
    ))])

fig.update_layout(title_text=f"Taxonomic Distribution for Sample {sample_id}", font_size=10)
fig.show()

       taxid         phylum            class          order  \
0    2036908     Ascomycota    Leotiomycetes     Helotiales   
1    2860823     Ascomycota    Leotiomycetes     Helotiales   
2    2867042     Ascomycota    Leotiomycetes     Helotiales   
3    2211668     Ascomycota    Leotiomycetes     Helotiales   
4    2086344     Ascomycota    Leotiomycetes    Erysiphales   
..       ...            ...              ...            ...   
778  1709935     Ascomycota  Dothideomycetes            NaN   
779  3113583  Basidiomycota   Agaricomycetes     Agaricales   
780  1307805     Ascomycota  Lecanoromycetes  Acarosporales   
781    73518     Ascomycota    Pichiomycetes      Serinales   
782    75750     Ascomycota   Eurotiomycetes     Eurotiales   

                family          genus              species  \
0      Rutstroemiaceae    Clarireedia            jacksonii   
1      Rutstroemiaceae    Clarireedia              paspali   
2      Rutstroemiaceae    Clarireedia           hainanens

In [39]:
# Define the sample ID
sample_id = '13114.angenent.65.s002'

# Calculate the total number of reads for the sample
total_reads_sample = yeast_abundance_df[sample_id].sum()

print(f"Total number of reads for sample {sample_id}: {total_reads_sample}")

# Calculate the total number of reads per phylum for the sample
# First, group by 'phylum' and sum the abundance for the specific sample
phylum_abundance_sample = yeast_abundance_df.groupby('phylum')[sample_id].sum()

print(f"\nTotal number of reads per phylum for sample {sample_id}:")
display(phylum_abundance_sample)

# Calculate and print the sum of reads per phylum
sum_of_phylum_reads = phylum_abundance_sample.sum()
print(f"\nSum of reads per phylum for sample {sample_id}: {sum_of_phylum_reads}")

Total number of reads for sample 13114.angenent.65.s002: 30233.0

Total number of reads per phylum for sample 13114.angenent.65.s002:


,13114.angenent.65.s002
phylum,
Ascomycota,28863.0
Basidiomycota,374.0
Blastocladiomycota,0.0
Chytridiomycota,0.0
Microsporidia,0.0
Mucoromycota,996.0
Zoopagomycota,0.0



Sum of reads per phylum for sample 13114.angenent.65.s002: 30233.0


In [42]:
# List of sample IDs to analyze
sample_ids_to_analyze = ['13114.distel.72.s014', '13114.distel.72.s015', '13114.rohwer.86.s001', '13114.rohwer.86.s003']

# Ensure merged_df is available (assuming it was created in a previous cell)
if 'merged_df' not in locals():
    print("Error: merged_df not found. Please run the cell that creates merged_df first.")
else:
    for sample_id in sample_ids_to_analyze:
        print(f"\nAnalyzing sample: {sample_id}")

        # Select the data for the specific sample from merged_df
        if sample_id in merged_df.columns:
            sample_data = merged_df[['Phylum', sample_id]].copy()

            # Filter out rows where the abundance for the sample is 0 or NaN
            sample_data = sample_data[sample_data[sample_id] > 0].dropna(subset=[sample_id])

            # Calculate the total number of reads per phylum for the sample
            # Group by 'Phylum' and sum the abundance for the specific sample
            phylum_abundance_sample = sample_data.groupby('Phylum')[sample_id].sum()

            print(f"Total number of reads per phylum for sample {sample_id}:")
            display(phylum_abundance_sample)

            # Calculate and print the sum of reads per phylum
            sum_of_phylum_reads = phylum_abundance_sample.sum()
            print(f"Sum of reads per phylum for sample {sample_id}: {sum_of_phylum_reads}")
        else:
            print(f"Sample ID '{sample_id}' not found in merged_df columns.")

Error: merged_df not found. Please run the cell that creates merged_df first.


## Direct from biom

In [44]:
from skbio import Table
HOME = '/content/drive/MyDrive/sci-kit-bio_local/FUNGI'

In [36]:
# !ls -lr $HOME

Convert jason file to hdf5 format

In [37]:
# !pip install -q biom-format
#!biom convert -i {HOME}/earth_microbiomes_K2_B_converted.json -o {HOME}/earth_microbiomes_K2_B_reconverted.biom --to-hdf5

In [45]:
from skbio import Table

# Load the BIOM file
biom_file_path = f'{HOME}/bioms/earth_microbiomes_K2_B_reconverted.biom'
table = Table.read(biom_file_path, 'biom')

# Display the table summary
print(table.shape)
print(table.ids()[1:5])
#print(table.ids(axis='observation'))

(783, 750)
['13114.angenent.65.s002' '13114.angenent.65.s003'
 '13114.angenent.65.s004' '13114.angenent.65.s005']


In [47]:
# Define the sample ID
sample_id = '13114.rohwer.86.s001'
#13114.distel.72.s014

# Get the abundance data for the specific sample from the table
sample_abundance = table.data(sample_id)

# Calculate the sum of abundances for the sample
total_reads_sample = sample_abundance.sum()

print(f"The sum of all values in column '{sample_id}' in the table is: {total_reads_sample}")

The sum of all values in column '13114.rohwer.86.s001' in the table is: 0.0


In [48]:
# Convert the skbio.Table to a pandas DataFrame
table_df = table.to_dataframe()

# Display the head of the DataFrame
print("Table as DataFrame:")
display(table_df.head())

Table as DataFrame:


,13114.angenent.65.s001,13114.angenent.65.s002,13114.angenent.65.s003,13114.angenent.65.s004,13114.angenent.65.s005,13114.angenent.65.s006,13114.angenent.65.s007,13114.angenent.65.s008,13114.angenent.65.s009,13114.berry.2.s001,...,13114.zaneveld.9.s013,13114.zaneveld.9.s014,13114.zaneveld.9.s015,13114.zaneveld.9.s016,13114.zaneveld.9.s017,13114.zaneveld.9.s018,13114.zaneveld.9.s019,13114.zaneveld.9.s020,13114.zaneveld.9.s021,13114.zaneveld.9.s022
2036908,8922.0,0,0,0,0,0,101.0,0,0,0,...,0,0,0,0,0,0,0,18631.0,0,0
2860823,4241.0,0,0,0,231.0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2867042,3223.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,2525.0,0,0
2211668,473.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,4849.0,0,5791.0,0,0
2086344,9845.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [49]:
dict(table.metadata('4934', 'observation'))

{'taxonomy': ['k__',
  'p__Ascomycota',
  'c__Saccharomycetes',
  'o__Saccharomycetales',
  'f__Saccharomycetaceae',
  'g__Lachancea',
  's__kluyveri']}

In [ ]:
# Get taxonomy

In [50]:
# Get observation metadata
obs_metadata = table.metadata(axis='observation')
obs_ids = table.ids(axis='observation')

# List to store parsed taxonomy data
parsed_taxonomy_list = []

# Iterate through observation metadata and parse taxonomy
for item in obs_metadata:
    taxonomy_ranks = {}
    if 'taxonomy' in item and isinstance(item['taxonomy'], list):
        # Assuming taxonomy is in the format k__Kingdom; p__Phylum; ...
        for rank_info in item['taxonomy']:
            if '__' in rank_info:
                rank_parts = rank_info.split('__')
                if len(rank_parts) == 2:
                    rank_level = rank_parts[0].strip()
                    rank_name = rank_parts[1].strip()
                    # Map shorthand to full names (optional, adjust as needed)
                    rank_mapping = {'k': 'Kingdom', 'p': 'Phylum', 'c': 'Class', 'o': 'Order', 'f': 'Family', 'g': 'Genus', 's': 'Species'}
                    full_rank_name = rank_mapping.get(rank_level, rank_level)
                    taxonomy_ranks[full_rank_name] = rank_name
            else:
                 # Handle cases where taxonomy is just the name without rank prefix
                 # You might need to adjust this based on your specific data format
                 pass # Or add logic to infer rank if possible

    parsed_taxonomy_list.append(taxonomy_ranks)

# Create a DataFrame from the parsed taxonomy data
taxonomy_df = pd.DataFrame(parsed_taxonomy_list, index=obs_ids)

# Display the head of the taxonomy DataFrame
print("Taxonomy DataFrame:")
display(taxonomy_df.head())

Taxonomy DataFrame:


,Kingdom,Phylum,Class,Order,Family,Genus,Species
2036908,,Ascomycota,Leotiomycetes,Helotiales,Rutstroemiaceae,Clarireedia,jacksonii
2860823,,Ascomycota,Leotiomycetes,Helotiales,Rutstroemiaceae,Clarireedia,paspali
2867042,,Ascomycota,Leotiomycetes,Helotiales,Rutstroemiaceae,Clarireedia,hainanense
2211668,,Ascomycota,Leotiomycetes,Helotiales,Rutstroemiaceae,Clarireedia,monteithiana
2086344,,Ascomycota,Leotiomycetes,Erysiphales,Erysiphaceae,Podosphaera,cerasi


In [51]:
# Merge the abundance table DataFrame with the taxonomy DataFrame
merged_df = table_df.merge(taxonomy_df, left_index=True, right_index=True, how='left')

# Display the head of the merged DataFrame
print("Merged Abundance and Taxonomy DataFrame:")
display(merged_df.head())

Merged Abundance and Taxonomy DataFrame:


,13114.angenent.65.s001,13114.angenent.65.s002,13114.angenent.65.s003,13114.angenent.65.s004,13114.angenent.65.s005,13114.angenent.65.s006,13114.angenent.65.s007,13114.angenent.65.s008,13114.angenent.65.s009,13114.berry.2.s001,...,13114.zaneveld.9.s020,13114.zaneveld.9.s021,13114.zaneveld.9.s022,Kingdom,Phylum,Class,Order,Family,Genus,Species
2036908,8922.0,0,0,0,0,0,101.0,0,0,0,...,18631.0,0,0,,Ascomycota,Leotiomycetes,Helotiales,Rutstroemiaceae,Clarireedia,jacksonii
2860823,4241.0,0,0,0,231.0,0,0,0,0,0,...,0,0,0,,Ascomycota,Leotiomycetes,Helotiales,Rutstroemiaceae,Clarireedia,paspali
2867042,3223.0,0,0,0,0,0,0,0,0,0,...,2525.0,0,0,,Ascomycota,Leotiomycetes,Helotiales,Rutstroemiaceae,Clarireedia,hainanense
2211668,473.0,0,0,0,0,0,0,0,0,0,...,5791.0,0,0,,Ascomycota,Leotiomycetes,Helotiales,Rutstroemiaceae,Clarireedia,monteithiana
2086344,9845.0,0,0,0,0,0,0,0,0,0,...,0,0,0,,Ascomycota,Leotiomycetes,Erysiphales,Erysiphaceae,Podosphaera,cerasi


In [52]:
# Define the sample ID
sample_id = '13114.rohwer.86.s001'

# Calculate the total number of reads for the sample from table_df
total_reads_sample = table_df[sample_id].sum()

print(f"Total number of reads for sample {sample_id} from table_df: {total_reads_sample}")

# To get reads per phylum from table_df, we need to merge with taxonomy information
# We can use the taxonomy_df created in cell 6dnHIS40DT-N
table_with_taxonomy = table_df.merge(taxonomy_df, left_index=True, right_index=True, how='left')

# Calculate the total number of reads per phylum for the sample
# Group by 'Phylum' and sum the abundance for the specific sample
phylum_abundance_sample = table_with_taxonomy.groupby('Phylum')[sample_id].sum()

print(f"\nTotal number of reads per phylum for sample {sample_id} from table_df:")
display(phylum_abundance_sample)

# Calculate and print the sum of reads per phylum
sum_of_phylum_reads = phylum_abundance_sample.sum()
print(f"\nSum of reads per phylum for sample {sample_id} from table_df: {sum_of_phylum_reads}")

Total number of reads for sample 13114.rohwer.86.s001 from table_df: 0.0

Total number of reads per phylum for sample 13114.rohwer.86.s001 from table_df:


,13114.rohwer.86.s001
Phylum,
,0
Ascomycota,0
Basidiomycota,0
Blastocladiomycota,0
Chytridiomycota,0
Microsporidia,0
Mucoromycota,0
Zoopagomycota,0



Sum of reads per phylum for sample 13114.rohwer.86.s001 from table_df: 0.0


Saving taxonomy by sample! (abundance table)

In [54]:
merged_df.head()

,13114.angenent.65.s001,13114.angenent.65.s002,13114.angenent.65.s003,13114.angenent.65.s004,13114.angenent.65.s005,13114.angenent.65.s006,13114.angenent.65.s007,13114.angenent.65.s008,13114.angenent.65.s009,13114.berry.2.s001,...,13114.zaneveld.9.s020,13114.zaneveld.9.s021,13114.zaneveld.9.s022,Kingdom,Phylum,Class,Order,Family,Genus,Species
2036908,8922.0,0,0,0,0,0,101.0,0,0,0,...,18631.0,0,0,,Ascomycota,Leotiomycetes,Helotiales,Rutstroemiaceae,Clarireedia,jacksonii
2860823,4241.0,0,0,0,231.0,0,0,0,0,0,...,0,0,0,,Ascomycota,Leotiomycetes,Helotiales,Rutstroemiaceae,Clarireedia,paspali
2867042,3223.0,0,0,0,0,0,0,0,0,0,...,2525.0,0,0,,Ascomycota,Leotiomycetes,Helotiales,Rutstroemiaceae,Clarireedia,hainanense
2211668,473.0,0,0,0,0,0,0,0,0,0,...,5791.0,0,0,,Ascomycota,Leotiomycetes,Helotiales,Rutstroemiaceae,Clarireedia,monteithiana
2086344,9845.0,0,0,0,0,0,0,0,0,0,...,0,0,0,,Ascomycota,Leotiomycetes,Erysiphales,Erysiphaceae,Podosphaera,cerasi


In [53]:
# Define the output file path
output_file_path = f'{HOME}/merged_abundance_taxonomy.csv'

# Print the merged_df DataFrame to a CSV file
merged_df.to_csv(output_file_path)

print(f"DataFrame successfully saved to {output_file_path}")

DataFrame successfully saved to /content/drive/MyDrive/sci-kit-bio_local/FUNGI/merged_abundance_taxonomy.csv
